## aai-511-final-project-group1


In [0]:
# Import required libraries


## 1. Data Collection: 

Data is collected and provided to you.



In [0]:
# Install kagglehub for dataset download
%pip install kagglehub -q

import kagglehub
import os
from pathlib import Path

print("="*80)
print("DOWNLOADING MIDI CLASSIC MUSIC DATASET FROM KAGGLE")
print("="*80)

# Download latest version of the dataset
print("\nDownloading dataset from Kaggle...")
path = kagglehub.dataset_download("blanderbuss/midi-classic-music")

print(f"\n✓ Dataset downloaded successfully!")
print(f"Path to dataset files: {path}")

# Set the dataset path for later use
dataset_path = path

# List all files and directories in the dataset path
print(f"\nExploring dataset structure...")
print("="*80)

try:
    # List items in the dataset directory
    items = os.listdir(dataset_path)
    print(f"\nFound {len(items)} items in the dataset directory:\n")
    
    for item in sorted(items)[:15]:
        item_path = os.path.join(dataset_path, item)
        item_type = "DIR" if os.path.isdir(item_path) else "FILE"
        print(f"  [{item_type}] {item}")
    
    if len(items) > 15:
        print(f"\n... and {len(items) - 15} more items")
        
except Exception as e:
    print(f"Error accessing dataset: {e}")

print("\n" + "="*80)


In [0]:
# Explore midiclassics.zip file - 4 Composers
import os
import zipfile
import shutil
from collections import defaultdict

print("="*80)
print("EXPLORING MIDICLASSICS.ZIP FILE")
print("="*80)

# Define the 4 composers we want to find
target_composers = ['bach', 'beethoven', 'chopin', 'mozart']

print(f"\nTarget composers: {', '.join([c.title() for c in target_composers])}")

# Look for midiclassics.zip in the dataset path
zip_file_path = os.path.join(dataset_path, 'midiclassics.zip')

print(f"\nLooking for zip file at: {zip_file_path}")

if not os.path.exists(zip_file_path):
    print(f"\n✗ midiclassics.zip not found at {zip_file_path}")
    print("\nSearching for zip files in dataset directory...")
    
    # Search for any zip files
    zip_files = [f for f in os.listdir(dataset_path) if f.endswith('.zip')]
    
    if zip_files:
        print(f"\nFound {len(zip_files)} zip file(s):")
        for zf in zip_files:
            print(f"  • {zf}")
        
        # Use the first zip file found
        zip_file_path = os.path.join(dataset_path, zip_files[0])
        print(f"\nUsing: {zip_files[0]}")
    else:
        print("\n✗ No zip files found in dataset directory")
        raise FileNotFoundError("midiclassics.zip not found")

print("\n" + "="*80)
print("ANALYZING ZIP FILE CONTENTS")
print("="*80)

# Explore the zip file contents
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    # Get all file names in the zip
    all_files = zip_ref.namelist()
    
    print(f"\nTotal files in zip: {len(all_files)}")
    
    # Organize files by composer
    composer_files = defaultdict(list)
    all_composers_found = set()
    
    for file_path in all_files:
        # Skip directories
        if file_path.endswith('/'):
            continue
        
        # Check if it's a MIDI file
        if file_path.lower().endswith(('.mid', '.midi')):
            # Extract composer name from path
            path_parts = file_path.split('/')
            
            # Try to find composer name in the path
            for part in path_parts:
                part_lower = part.lower()
                all_composers_found.add(part)
                
                # Check if this part matches any target composer
                for composer in target_composers:
                    if composer in part_lower:
                        composer_files[composer].append(file_path)
                        break
    
    print("\n" + "="*80)
    print("ALL COMPOSERS/FOLDERS FOUND IN ZIP")
    print("="*80)
    
    # Show all unique folder names
    folders = set()
    for file_path in all_files:
        parts = file_path.split('/')
        if len(parts) > 1:
            folders.add(parts[0])
    
    for folder in sorted(folders):
        marker = "✓" if any(tc in folder.lower() for tc in target_composers) else "✗"
        print(f"  [{marker}] {folder}")
    
    # Display statistics for target composers
    print("\n" + "="*80)
    print("MIDI FILES FOR TARGET COMPOSERS")
    print("="*80)
    
    composer_stats = {}
    for composer in target_composers:
        if composer in composer_files:
            midi_count = len(composer_files[composer])
            composer_stats[composer] = {
                'files': composer_files[composer],
                'midi_count': midi_count
            }
            
            print(f"\n{composer.title()}:")
            print(f"  MIDI files: {midi_count}")
            print(f"  Sample files:")
            for file_path in composer_files[composer][:3]:
                print(f"    • {file_path}")
            if midi_count > 3:
                print(f"    ... and {midi_count - 3} more")
        else:
            print(f"\n{composer.title()}: NOT FOUND")
    
    # Extract the zip file to a temporary directory
    extract_path = os.path.join(dataset_path, 'extracted_midi')
    
    print("\n" + "="*80)
    print("EXTRACTING ZIP FILE")
    print("="*80)
    
    print(f"\nExtracting to: {extract_path}")
    
    # Create extraction directory if it doesn't exist
    os.makedirs(extract_path, exist_ok=True)
    
    # Extract all files
    zip_ref.extractall(extract_path)
    
    print("✓ Extraction complete!")
    
    # Update dataset_path to point to extracted location
    dataset_path = extract_path

# Summary
total_files = sum([stats['midi_count'] for stats in composer_stats.values()])
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"\nTotal MIDI files (4 composers): {total_files}")
print(f"Composers found: {len(composer_stats)}")
print(f"Extracted to: {dataset_path}")

print("\n" + "="*80)
print("ZIP FILE EXPLORATION COMPLETE!")
print("="*80)

## 2.1 Exploratory Data Analysis (EDA)

EDA examines the statistical properties and patterns of MIDI files across 9 composers (Bach, Bartok, Byrd, Chopin, Handel, Hummel, Mendelssohn, Mozart, and Schumann), analyzing duration, tempo, complexity, and pitch characteristics to identify distinguishing features for classification. This analysis reveals data quality issues, informs feature selection for the deep learning model, and guides preprocessing decisions.



In [0]:
# Install required library for MIDI processing
%pip install pretty_midi

In [0]:
# Restart Python interpreter to apply new library installations
dbutils.library.restartPython()

In [0]:
# Exploratory Data Analysis on 4 Composers
import pretty_midi
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("="*80)
print("EXPLORATORY DATA ANALYSIS (EDA) - 4 COMPOSERS")
print("="*80)

# Define the 4 composers
composers = ['bach', 'beethoven', 'chopin', 'mozart']

print(f"\nAnalyzing: {', '.join([c.title() for c in composers])}")
print("\n" + "="*80)

# Initialize data structures for EDA
composer_stats = []

# Analyze files from each composer
for composer in composers:
    # Find the matching folder (case-insensitive)
    all_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    matching_folders = [f for f in all_folders if f.lower() == composer.lower()]
    
    if not matching_folders:
        print(f"✗ {composer.title()}: Folder not found")
        continue
    
    composer_folder = matching_folders[0]
    composer_path = os.path.join(dataset_path, composer_folder)
    
    try:
        # Get all MIDI files for this composer
        midi_files = [f for f in os.listdir(composer_path) 
                     if f.endswith('.mid') or f.endswith('.midi')]
        
        print(f"\n{composer.title()}: Analyzing {len(midi_files)} MIDI files...")
        
        # Analyze files from this composer (sample first 10 for quick EDA)
        files_processed = 0
        for midi_file in midi_files[:10]:
            try:
                midi_file_path = os.path.join(composer_path, midi_file)
                
                # Load MIDI file
                midi_data = pretty_midi.PrettyMIDI(midi_file_path)
                
                # Extract features
                stats = {
                    'composer': composer,
                    'file_name': midi_file,
                    'duration_sec': midi_data.get_end_time(),
                    'tempo_estimate': midi_data.estimate_tempo(),
                    'num_instruments': len(midi_data.instruments)
                }
                
                # Count total notes across all instruments
                total_notes = sum([len(instrument.notes) for instrument in midi_data.instruments])
                stats['total_notes'] = total_notes
                
                # Get pitch statistics
                if midi_data.instruments:
                    pitches = [note.pitch for instrument in midi_data.instruments 
                              for note in instrument.notes if not instrument.is_drum]
                    if pitches:
                        stats['pitch_mean'] = np.mean(pitches)
                        stats['pitch_std'] = np.std(pitches)
                        stats['pitch_range'] = max(pitches) - min(pitches)
                        stats['pitch_min'] = min(pitches)
                        stats['pitch_max'] = max(pitches)
                    else:
                        stats['pitch_mean'] = 0
                        stats['pitch_std'] = 0
                        stats['pitch_range'] = 0
                        stats['pitch_min'] = 0
                        stats['pitch_max'] = 0
                else:
                    stats['pitch_mean'] = 0
                    stats['pitch_std'] = 0
                    stats['pitch_range'] = 0
                    stats['pitch_min'] = 0
                    stats['pitch_max'] = 0
                
                composer_stats.append(stats)
                files_processed += 1
                    
            except Exception as e:
                print(f"  ⚠ Error processing {midi_file}: {str(e)[:50]}...")
                continue
        
        print(f"  ✓ Successfully processed {files_processed} files")
        
    except Exception as e:
        print(f"  ✗ Error accessing {composer}: {e}")

print("\n" + "="*80)
print(f"TOTAL FILES ANALYZED: {len(composer_stats)}")
print("="*80)

# Create DataFrame and display summary statistics
if len(composer_stats) > 0:
    df_stats = pd.DataFrame(composer_stats)
    
    print("\nSummary Statistics by Composer:")
    print("-" * 80)
    
    summary = df_stats.groupby('composer').agg({
        'duration_sec': ['mean', 'std', 'min', 'max'],
        'tempo_estimate': ['mean', 'std', 'min', 'max'],
        'total_notes': ['mean', 'std', 'min', 'max'],
        'num_instruments': ['mean', 'max'],
        'pitch_mean': ['mean', 'std'],
        'pitch_range': ['mean', 'max']
    }).round(2)
    
    display(summary)
    
    print("\nOverall Dataset Statistics:")
    print(f"  Average duration: {df_stats['duration_sec'].mean():.2f} seconds")
    print(f"  Average tempo: {df_stats['tempo_estimate'].mean():.2f} BPM")
    print(f"  Average notes per file: {df_stats['total_notes'].mean():.0f}")
    print(f"  Average pitch: {df_stats['pitch_mean'].mean():.2f} (MIDI note number)")
    print(f"  Average instruments: {df_stats['num_instruments'].mean():.2f}")
    
    print("\n" + "="*80)
else:
    print("\n⚠ WARNING: No files were successfully processed. Check file paths and MIDI file format.")
    print("="*80)

In [0]:
# Create visualizations for EDA - 4 Composers
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("="*80)
print("CREATING EDA VISUALIZATIONS")
print("="*80)

# Set up the plotting style
sns.set_style("whitegrid")
sns.set_palette("husl")

# Create a figure with multiple subplots
fig = plt.figure(figsize=(18, 12))

# 1. Average Duration by Composer
ax1 = plt.subplot(2, 3, 1)
df_duration = df_stats.groupby('composer')['duration_sec'].mean().sort_values(ascending=False)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
sns.barplot(x=df_duration.values, y=df_duration.index, ax=ax1, palette=colors)
ax1.set_xlabel('Average Duration (seconds)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Composer', fontsize=12, fontweight='bold')
ax1.set_title('Average Duration by Composer', fontsize=14, fontweight='bold')
for i, v in enumerate(df_duration.values):
    ax1.text(v + 5, i, f'{v:.1f}s', va='center', fontweight='bold')

# 2. Average Tempo by Composer
ax2 = plt.subplot(2, 3, 2)
df_tempo = df_stats.groupby('composer')['tempo_estimate'].mean().sort_values(ascending=False)
sns.barplot(x=df_tempo.values, y=df_tempo.index, ax=ax2, palette=colors)
ax2.set_xlabel('Average Tempo (BPM)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Composer', fontsize=12, fontweight='bold')
ax2.set_title('Average Tempo by Composer', fontsize=14, fontweight='bold')
for i, v in enumerate(df_tempo.values):
    ax2.text(v + 3, i, f'{v:.1f}', va='center', fontweight='bold')

# 3. Average Total Notes by Composer
ax3 = plt.subplot(2, 3, 3)
df_notes = df_stats.groupby('composer')['total_notes'].mean().sort_values(ascending=False)
sns.barplot(x=df_notes.values, y=df_notes.index, ax=ax3, palette=colors)
ax3.set_xlabel('Average Total Notes', fontsize=12, fontweight='bold')
ax3.set_ylabel('Composer', fontsize=12, fontweight='bold')
ax3.set_title('Average Total Notes by Composer', fontsize=14, fontweight='bold')
for i, v in enumerate(df_notes.values):
    ax3.text(v + 100, i, f'{int(v)}', va='center', fontweight='bold')

# 4. Box Plot - Duration Distribution
ax4 = plt.subplot(2, 3, 4)
sns.boxplot(data=df_stats, y='composer', x='duration_sec', ax=ax4, palette=colors)
ax4.set_xlabel('Duration (seconds)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Composer', fontsize=12, fontweight='bold')
ax4.set_title('Duration Distribution by Composer', fontsize=14, fontweight='bold')

# 5. Box Plot - Tempo Distribution
ax5 = plt.subplot(2, 3, 5)
sns.boxplot(data=df_stats, y='composer', x='tempo_estimate', ax=ax5, palette=colors)
ax5.set_xlabel('Tempo (BPM)', fontsize=12, fontweight='bold')
ax5.set_ylabel('Composer', fontsize=12, fontweight='bold')
ax5.set_title('Tempo Distribution by Composer', fontsize=14, fontweight='bold')

# 6. Scatter Plot - Pitch Mean vs Range
ax6 = plt.subplot(2, 3, 6)
for idx, composer in enumerate(sorted(df_stats['composer'].unique())):
    composer_data = df_stats[df_stats['composer'] == composer]
    ax6.scatter(composer_data['pitch_mean'], composer_data['pitch_range'], 
                label=composer.title(), s=120, alpha=0.7, color=colors[idx])
ax6.set_xlabel('Pitch Mean (MIDI Note)', fontsize=12, fontweight='bold')
ax6.set_ylabel('Pitch Range (Semitones)', fontsize=12, fontweight='bold')
ax6.set_title('Pitch Mean vs Range by Composer', fontsize=14, fontweight='bold')
ax6.legend(fontsize=10, loc='best')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Second figure - Correlation heatmap and additional analysis
print("\nCreating correlation analysis and distribution plots...")

fig2 = plt.figure(figsize=(16, 6))

# 1. Correlation Heatmap
ax7 = plt.subplot(1, 2, 1)
numeric_cols = ['duration_sec', 'tempo_estimate', 'total_notes', 'pitch_mean', 'pitch_std', 'pitch_range']
corr_matrix = df_stats[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=2, cbar_kws={"shrink": 0.8}, ax=ax7, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
ax7.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
ax7.set_xticklabels(ax7.get_xticklabels(), rotation=45, ha='right')
ax7.set_yticklabels(ax7.get_yticklabels(), rotation=0)

# 2. Violin Plot - Total Notes Distribution
ax8 = plt.subplot(1, 2, 2)
sns.violinplot(data=df_stats, y='composer', x='total_notes', ax=ax8, palette=colors)
ax8.set_xlabel('Total Notes', fontsize=12, fontweight='bold')
ax8.set_ylabel('Composer', fontsize=12, fontweight='bold')
ax8.set_title('Total Notes Distribution by Composer', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("EDA VISUALIZATIONS COMPLETE!")
print("="*80)

print("\nKey Observations:")
print("-" * 80)
print(f"  • Longest pieces: {df_duration.index[0].title()} ({df_duration.values[0]:.1f}s average)")
print(f"  • Fastest tempo: {df_tempo.index[0].title()} ({df_tempo.values[0]:.1f} BPM average)")
print(f"  • Most complex: {df_notes.index[0].title()} ({int(df_notes.values[0])} notes average)")
print(f"  • Strongest correlation: Duration & Total Notes (r={corr_matrix.loc['duration_sec', 'total_notes']:.2f})")

print("\n" + "="*80)

## 2.2 Data Pre-processing: 

Convert the musical scores into a format suitable for deep learning models. This involves converting the musical scores into MIDI files and applying data augmentation techniques.

In [0]:
# Data Pre-Processing: Convert MIDI to sequences and apply augmentation - 4 Composers
import pretty_midi
import numpy as np
import pandas as pd
from collections import Counter
import os
import random
from sklearn.model_selection import train_test_split

print("="*80)
print("MIDI DATA PRE-PROCESSING - 4 COMPOSERS")
print("="*80)

# Configuration
SEQUENCE_LENGTH = 100  # Number of notes in each sequence
TRAIN_SPLIT = 0.7      # 70% training
DEV_SPLIT = 0.15       # 15% development
TEST_SPLIT = 0.15      # 15% test

# Create composer to label mapping (4 composers only)
composers = ['bach', 'beethoven', 'chopin', 'mozart']
composer_to_label = {composer: idx for idx, composer in enumerate(composers)}
label_to_composer = {idx: composer for composer, idx in composer_to_label.items()}

print(f"\nComposer Label Mapping:")
for composer, label in composer_to_label.items():
    print(f"  {label}: {composer.title()}")

def extract_notes_from_midi(midi_path):
    try:
        midi_data = pretty_midi.PrettyMIDI(midi_path)
        notes = []
        for instrument in midi_data.instruments:
            if not instrument.is_drum:
                for note in instrument.notes:
                    notes.append({
                        'pitch': note.pitch,
                        'start': note.start,
                        'end': note.end,
                        'duration': note.end - note.start,
                        'velocity': note.velocity
                    })
        notes = sorted(notes, key=lambda x: x['start'])
        return notes
    except:
        return None

def notes_to_sequences(notes, sequence_length=100):
    if not notes or len(notes) < sequence_length:
        return []
    sequences = []
    for i in range(0, len(notes) - sequence_length + 1, sequence_length // 2):
        seq = notes[i:i + sequence_length]
        sequence_array = np.array([
            [n['pitch'] / 127.0,
             min(n['duration'], 4.0) / 4.0,
             n['velocity'] / 127.0]
            for n in seq
        ])
        sequences.append(sequence_array)
    return sequences

# Process all composers
print("\n" + "="*80)
print("EXTRACTING SEQUENCES FROM ALL COMPOSERS")
print("="*80)

all_sequences = []
all_labels = []

for composer in composers:
    all_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    matching_folders = [f for f in all_folders if f.lower() == composer.lower()]
    
    if not matching_folders:
        print(f"\n✗ {composer.title()}: Folder not found")
        continue
    
    composer_folder = matching_folders[0]
    composer_path = os.path.join(dataset_path, composer_folder)
    
    midi_files = [f for f in os.listdir(composer_path) if f.endswith('.mid') or f.endswith('.midi')]
    print(f"\n{composer.title()}: Processing {len(midi_files)} MIDI files...")
    
    composer_sequences = 0
    for midi_file in midi_files:
        midi_file_path = os.path.join(composer_path, midi_file)
        notes = extract_notes_from_midi(midi_file_path)
        
        if notes:
            sequences = notes_to_sequences(notes, SEQUENCE_LENGTH)
            if sequences:
                all_sequences.extend(sequences)
                all_labels.extend([composer_to_label[composer]] * len(sequences))
                composer_sequences += len(sequences)
    
    print(f"  ✓ Extracted {composer_sequences} sequences")

print("\n" + "="*80)
print(f"TOTAL SEQUENCES: {len(all_sequences)}")
print("="*80)

# Convert to arrays and split
X_all = np.array(all_sequences)
y_all = np.array(all_labels)

print("\nSplitting data...")
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Simple augmentation for training set
print("\nApplying augmentation...")
augmented_X = []
augmented_y = []
for i in range(len(X_train)):
    augmented_X.append(X_train[i])
    augmented_y.append(y_train[i])
    # Pitch shift
    for shift in [-0.02, 0.02]:
        aug = X_train[i].copy()
        aug[:, 0] = np.clip(aug[:, 0] + shift, 0, 1)
        augmented_X.append(aug)
        augmented_y.append(y_train[i])

X_train = np.array(augmented_X)
y_train = np.array(augmented_y)

print("\n" + "="*80)
print("DATA PRE-PROCESSING COMPLETE")
print("="*80)
print(f"\nTraining: {X_train.shape} ({len(X_train)} sequences)")
print(f"Dev: {X_dev.shape} ({len(X_dev)} sequences)")
print(f"Test: {X_test.shape} ({len(X_test)} sequences)")

print("\nClass Distribution (Training):")
train_dist = Counter(y_train)
for label in sorted(train_dist.keys()):
    print(f"  {label_to_composer[label].title()}: {train_dist[label]}")

print("\n" + "="*80)

## 3. Feature Extraction: 

Extract features from the MIDI files, such as notes, chords, and tempo, using music analysis tools.

In [0]:
# Feature Extraction: Prepare FULL Piano Rolls for CNN + Summary Statistics
import pretty_midi
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("="*80)
print("FEATURE EXTRACTION - PIANO ROLLS FOR CNN + STATISTICS")
print("="*80)

# Configuration for CNN
FRAME_RATE = 16  # Hz - frames per second
MIN_MIDI_PITCH = 21   # A0 - lowest note on piano
MAX_MIDI_PITCH = 108  # C8 - highest note on piano
PITCH_RANGE = MAX_MIDI_PITCH - MIN_MIDI_PITCH + 1  # 88 keys
MAX_TIME_STEPS = 500  # Maximum time steps for CNN (pad/truncate)

composers = ['bach', 'beethoven', 'chopin', 'mozart']
composer_to_label = {composer: idx for idx, composer in enumerate(composers)}

print(f"\nConfiguration:")
print(f"  Frame Rate: {FRAME_RATE} Hz")
print(f"  Pitch Range: {MIN_MIDI_PITCH}-{MAX_MIDI_PITCH} ({PITCH_RANGE} keys)")
print(f"  Max Time Steps: {MAX_TIME_STEPS}")
print(f"  Composers: {', '.join([c.title() for c in composers])}")

def extract_piano_roll(midi_data, frame_rate=16, max_time_steps=500):
    """Extract piano roll representation for CNN models."""
    try:
        piano_roll_full = midi_data.get_piano_roll(fs=frame_rate)
        piano_roll = piano_roll_full[MIN_MIDI_PITCH:MAX_MIDI_PITCH+1, :].T
        piano_roll = piano_roll / 127.0  # Normalize to [0, 1]
        
        # Pad or truncate to fixed length
        if piano_roll.shape[0] < max_time_steps:
            # Pad with zeros
            padding = np.zeros((max_time_steps - piano_roll.shape[0], PITCH_RANGE))
            piano_roll = np.vstack([piano_roll, padding])
        else:
            # Truncate
            piano_roll = piano_roll[:max_time_steps, :]
        
        return piano_roll
    except:
        return None

# Extract piano rolls from ALL files for CNN training
print("\n" + "="*80)
print("EXTRACTING PIANO ROLLS FROM ALL FILES (FOR CNN)")
print("="*80)

all_piano_rolls = []
all_labels_cnn = []
file_metadata = []

for composer in composers:
    all_folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    matching_folders = [f for f in all_folders if f.lower() == composer.lower()]
    
    if not matching_folders:
        print(f"\n✗ {composer.title()}: Folder not found")
        continue
    
    composer_folder = matching_folders[0]
    composer_path = os.path.join(dataset_path, composer_folder)
    midi_files = [f for f in os.listdir(composer_path) if f.endswith('.mid') or f.endswith('.midi')]
    
    print(f"\n{composer.title()}: Processing {len(midi_files)} files...")
    
    processed = 0
    for midi_file in midi_files:
        try:
            midi_file_path = os.path.join(composer_path, midi_file)
            midi_data = pretty_midi.PrettyMIDI(midi_file_path)
            
            # Extract piano roll
            piano_roll = extract_piano_roll(midi_data, FRAME_RATE, MAX_TIME_STEPS)
            
            if piano_roll is not None and piano_roll.shape == (MAX_TIME_STEPS, PITCH_RANGE):
                all_piano_rolls.append(piano_roll)
                all_labels_cnn.append(composer_to_label[composer])
                
                # Store metadata for analysis
                file_metadata.append({
                    'composer': composer,
                    'file': midi_file,
                    'duration': midi_data.get_end_time(),
                    'tempo': midi_data.estimate_tempo(),
                    'n_instruments': len(midi_data.instruments)
                })
                
                processed += 1
                
        except Exception as e:
            continue
    
    print(f"  ✓ Extracted {processed} piano rolls")

print("\n" + "="*80)
print(f"TOTAL PIANO ROLLS EXTRACTED: {len(all_piano_rolls)}")
print("="*80)

# Convert to numpy arrays
X_cnn_all = np.array(all_piano_rolls)
y_cnn_all = np.array(all_labels_cnn)

print(f"\nPiano Roll Dataset Shape: {X_cnn_all.shape}")
print(f"  - Samples: {X_cnn_all.shape[0]}")
print(f"  - Time Steps: {X_cnn_all.shape[1]}")
print(f"  - Pitch Range (Piano Keys): {X_cnn_all.shape[2]}")

# Split into train/dev/test (same ratios as Cell 12)
print("\n" + "="*80)
print("SPLITTING CNN DATA (TRAIN/DEV/TEST)")
print("="*80)

X_train_cnn, X_temp_cnn, y_train_cnn, y_temp_cnn = train_test_split(
    X_cnn_all, y_cnn_all, test_size=0.3, random_state=42, stratify=y_cnn_all
)

X_dev_cnn, X_test_cnn, y_dev_cnn, y_test_cnn = train_test_split(
    X_temp_cnn, y_temp_cnn, test_size=0.5, random_state=42, stratify=y_temp_cnn
)

# Reshape for CNN: (samples, time_steps, features, channels)
X_train_cnn = X_train_cnn.reshape(-1, MAX_TIME_STEPS, PITCH_RANGE, 1)
X_dev_cnn = X_dev_cnn.reshape(-1, MAX_TIME_STEPS, PITCH_RANGE, 1)
X_test_cnn = X_test_cnn.reshape(-1, MAX_TIME_STEPS, PITCH_RANGE, 1)

print("\nCNN Data Shapes:")
print(f"  Training:   {X_train_cnn.shape}")
print(f"  Dev:        {X_dev_cnn.shape}")
print(f"  Test:       {X_test_cnn.shape}")

print("\nClass Distribution (CNN Training):")
from collections import Counter
train_dist_cnn = Counter(y_train_cnn)
label_to_composer = {idx: composer for composer, idx in composer_to_label.items()}
for label in sorted(train_dist_cnn.keys()):
    print(f"  {label_to_composer[label].title()}: {train_dist_cnn[label]} samples")

# Create summary statistics DataFrame
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

if file_metadata:
    df_features = pd.DataFrame(file_metadata)
    
    print("\nFeature Summary by Composer:")
    print("-" * 80)
    
    summary = df_features.groupby('composer').agg({
        'duration': ['mean', 'std', 'min', 'max'],
        'tempo': ['mean', 'std', 'min', 'max'],
        'n_instruments': ['mean', 'max']
    }).round(2)
    
    display(summary)

print("\n" + "="*80)
print("FEATURE EXTRACTION COMPLETE!")
print("="*80)

print("\n📊 Data Available for Model Building:")
print("-" * 80)
print("\n✅ LSTM/Transformer (from Cell 12):")
print(f"   - X_train: {X_train.shape} (sequences, notes, features)")
print(f"   - X_dev:   {X_dev.shape}")
print(f"   - X_test:  {X_test.shape}")

print("\n✅ CNN (from this cell):")
print(f"   - X_train_cnn: {X_train_cnn.shape} (samples, time, keys, channels)")
print(f"   - X_dev_cnn:   {X_dev_cnn.shape}")
print(f"   - X_test_cnn:  {X_test_cnn.shape}")

print("\n" + "="*80)
print("🎯 READY FOR MODEL BUILDING: LSTM, CNN, and Transformer!")
print("="*80)

4. Model Building: Develop a deep learning model using LSTM and CNN architectures to classify the musical scores according to the composer.

## 4.1 LSTM


In [0]:
# 

## 4.2 CNN

## 4.3 Transformer

5. Model Training: Train the deep learning model using the pre-processed and feature-extracted data.

In [0]:
# 

6. Model Evaluation: Evaluate the performance of the deep learning model using accuracy, precision, and recall metrics.

In [0]:
# 

7. Model Optimization: Optimize the deep learning model by fine-tuning hyperparameters.

In [0]:
# 